# Hannah: controlled LLM interpretation comparison

This notebook compares **Claude, Gemini, Qwen, and GLM** using exactly the same archived inversion and pseudo-geological model from `Hannah_Inversion_GPT`. It runs in `interpret_existing` mode: SimPEG inversion is never rerun, and the source directory remains read-only.

Controlled inputs for all four models:

- identical density and susceptibility models, mesh, topography, and predicted/observed data;
- identical archived Geo 1–5 pseudo-geological model;
- identical geological prior and target Geo IDs (Geo 4 and Geo 5);
- identical report request and reviewer checks;
- separate outputs under `Hannah_LLM_comparison/<MODEL>/`.


In [ ]:
import json
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

from multi_agent_runner import LLMClient, MultiAgentOrchestrator

ROOT = Path.cwd().resolve()
SOURCE_DIR = (ROOT / "Hannah_Inversion_GPT").resolve()
OUTPUT_ROOT = (ROOT / "Hannah_LLM_comparison").resolve()
GEOLOGY_CONTEXT = (ROOT / "Hannah" / "Hannah_geology_context.txt").resolve()

required_source_files = [
    SOURCE_DIR / "mesh" / "mesh_core.msh",
    SOURCE_DIR / "inversion_result" / "joint_density_core.npy",
    SOURCE_DIR / "inversion_result" / "joint_susceptibility_core.npy",
    SOURCE_DIR / "geology_models" / "unit_id_3d.npy",
    SOURCE_DIR / "geology_models" / "geo_id_3d.npy",
    SOURCE_DIR / "geology_models" / "geo_defs.json",
    SOURCE_DIR / "topo" / "topography.xyz",
]
missing = [str(path) for path in required_source_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required archived inputs:\n" + "\n".join(missing))
if not GEOLOGY_CONTEXT.is_file():
    raise FileNotFoundError(f"Missing shared geological prior: {GEOLOGY_CONTEXT}")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Archived source (read-only):", SOURCE_DIR)
print("Comparison output root     :", OUTPUT_ROOT)
print("Shared geological prior   :", GEOLOGY_CONTEXT)


## Model endpoints and credentials

Claude uses the Anthropic API directly, Gemini and Qwen use OpenRouter, and GLM uses OpenCode Zen. Enter the three API keys explicitly in the next cell. The notebook does not print or write them to output files.


In [ ]:
import os

# ==== API credentials from environment variables ====
# Set only the variables required by the models you intend to run.
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENCODE_API_KEY = os.environ.get("OPENCODE_API_KEY", "")

MODEL_SPECS = {
    "CLAUDE": {
        "base_url": "https://api.anthropic.com/v1/",
        "model": "claude-sonnet-4-5",
        "api_key": ANTHROPIC_API_KEY,
    },
    "GEMINI": {
        "base_url": "https://openrouter.ai/api/v1",
        "model": "google/gemini-2.5-pro",
        "api_key": OPENROUTER_API_KEY,
    },
    "QWEN": {
        "base_url": "https://openrouter.ai/api/v1",
        "model": "qwen/qwen3-vl-30b-a3b-thinking",
        "api_key": OPENROUTER_API_KEY,
    },
    "GLM": {
        "base_url": "https://opencode.ai/zen/go/v1",
        "model": "glm-5.1",
        "api_key": OPENCODE_API_KEY,
    },
}
MODELS_TO_RUN = ["CLAUDE", "GEMINI", "QWEN", "GLM"]

def validate_api_key(model_label):
    key = str(MODEL_SPECS[model_label]["api_key"]).strip()
    if not key:
        raise RuntimeError(
            f"Set the required API-key environment variable before running {model_label}."
        )
    return key

for label in MODELS_TO_RUN:
    spec = MODEL_SPECS[label]
    key_status = "configured" if str(spec["api_key"]).strip() else "missing"
    print(f"{label:7s} | {spec['model']} | {spec['base_url']} | key: {key_status}")


## Fixed interpretation configuration

The configuration below is passed directly to `run_from_config`, so configuration agents cannot alter the archived inversion settings. `reuse_existing_geology` also fixes the pseudo-geological classes; only the language-model interpretation, self-review, and report wording differ.


In [ ]:
COMMON_REQUEST = """
Interpret the existing Hannah joint gravity-magnetic inversion and its archived pseudo-geological model for natural-hydrogen exploration. Do not rerun or claim to have rerun the inversion. Use only the supplied numerical evidence, geological prior, target-depth audit, and archived figures. Treat model z as elevation in metres and calculate depth below surface only relative to local topography; never interpret a mesh-layer index as physical depth. Distinguish Unit IDs from Geo IDs. Evaluate the spatial geometry, physical-property support, geological plausibility, exploration significance, uncertainty, and depth-dependent confidence of the moderately and highly serpentinized target groups (Geo 4 and Geo 5). Give precise numbers only when they are supported by the structured evidence. Produce a detailed technical report in English.
""".strip()

BASE_CONFIG = {
    "project": {
        "name": "Hannah",
        "input_dir": str((ROOT / "Hannah").resolve()),
        "output_dir": str(SOURCE_DIR),
        "source_inversion_dir": str(SOURCE_DIR),
        "interpretation_output_dir": None,
    },
    "geology": {
        "mode": "reuse_existing_geology",
        "context_path": str(GEOLOGY_CONTEXT),
        "target_unit_ids": [],
        "target_geo_ids": [4, 5],
        "target_name": "Serpentinite hydrogen play along the Collayomi fault",
    },
    "run": {
        "execution_mode": "interpret_existing",
        "run_inversion": False,
        "run_geology_model": True,
        "make_plots": False,
        "reuse_existing_geology": True,
        "skip_configuration_agents": True,
        "write_reports": True,
        "review_enabled": True,
        "max_review_rounds": 1,
        "overwrite": True,
    },
}

assert BASE_CONFIG["run"]["execution_mode"] == "interpret_existing"
assert BASE_CONFIG["run"]["run_inversion"] is False
assert BASE_CONFIG["run"]["reuse_existing_geology"] is True
assert BASE_CONFIG["geology"]["target_geo_ids"] == [4, 5]
print(json.dumps(BASE_CONFIG, indent=2, ensure_ascii=False))


In [ ]:
def run_one_interpretation(model_label):
    spec = MODEL_SPECS[model_label]
    output_dir = (OUTPUT_ROOT / model_label).resolve()
    if output_dir == SOURCE_DIR or SOURCE_DIR in output_dir.parents:
        raise RuntimeError(f"Unsafe output path inside archived source: {output_dir}")

    cfg = deepcopy(BASE_CONFIG)
    cfg["project"]["interpretation_output_dir"] = str(output_dir)

    llm = LLMClient(
        api_key=validate_api_key(model_label),
        base_url=spec["base_url"],
        model=spec["model"],
    )
    orchestrator = MultiAgentOrchestrator(llm=llm, enable_vision=False)

    print(f"\n{'=' * 78}\n{model_label}: interpreting archived GPT inversion\n{'=' * 78}")
    result = orchestrator.run_from_config(cfg, user_request=COMMON_REQUEST)

    run_manifest_path = Path(result["workflow_result"]["interpretation_output_dir"]) / "run_manifest.json"
    run_manifest = json.loads(run_manifest_path.read_text(encoding="utf-8"))
    assert run_manifest["execution_mode"] == "interpret_existing"
    assert run_manifest["inversion_reused"] is True
    assert run_manifest["inversion_recomputed"] is False
    assert run_manifest["source_directory_read_only"] is True

    run_info = {
        "model_label": model_label,
        "model_name": spec["model"],
        "api_base": spec["base_url"],
        "source_inversion_dir": str(SOURCE_DIR),
        "interpretation_output_dir": str(output_dir),
        "execution_mode": "interpret_existing",
        "inversion_recomputed": False,
        "reused_existing_geology": True,
        "target_geo_ids": [4, 5],
        "geology_context": str(GEOLOGY_CONTEXT),
        "review_enabled": True,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "report_path": result.get("report_path"),
        "pdf_path": result.get("pdf_path"),
        "review_decision": (result.get("review") or {}).get("decision"),
    }
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "model_run_info.json").write_text(
        json.dumps(run_info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    return result, run_info


## Run the four controlled interpretations

Each model is isolated in its own output folder. A provider failure is recorded without discarding successful runs from the other providers. Rerunning this cell updates the same four folders; it never writes into `Hannah_Inversion_GPT`.


In [ ]:
results = {}
comparison_rows = []

for model_label in MODELS_TO_RUN:
    try:
        result, run_info = run_one_interpretation(model_label)
        results[model_label] = result
        comparison_rows.append({"status": "completed", **run_info})
        print(f"{model_label} completed: {run_info['report_path']}")
    except Exception as exc:
        comparison_rows.append({
            "model_label": model_label,
            "model_name": MODEL_SPECS[model_label]["model"],
            "status": "failed",
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        print(f"{model_label} FAILED: {type(exc).__name__}: {exc}")

comparison_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "comparison_design": "Same GPT inversion, same archived geology, same prior, same prompt; LLM differs.",
    "source_inversion_dir": str(SOURCE_DIR),
    "inversion_recomputed": False,
    "rows": comparison_rows,
}
manifest_path = OUTPUT_ROOT / "comparison_manifest.json"
manifest_path.write_text(json.dumps(comparison_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("\nComparison manifest:", manifest_path)


In [ ]:
# Compact completion table and final safety check.
for row in comparison_rows:
    label = row["model_label"]
    status = row["status"]
    report = row.get("report_path", "")
    review = row.get("review_decision", "")
    print(f"{label:7s} | {status:9s} | review={str(review):14s} | {report}")

completed = [row for row in comparison_rows if row["status"] == "completed"]
failed = [row for row in comparison_rows if row["status"] == "failed"]
for row in completed:
    assert row["inversion_recomputed"] is False
    assert Path(row["interpretation_output_dir"]).parent == OUTPUT_ROOT
print(f"\nCompleted {len(completed)}/{len(MODELS_TO_RUN)} interpretations; failed {len(failed)}.")
if failed:
    print("Failed models can be rerun after fixing their credential/endpoint issue; successful folders are already usable.")
